In [1]:
# Imports
import sys
from pathlib import Path

# Para que el notebook vea src/ desde notebooks/
sys.path.insert(0, str(Path("..").resolve()))

from src.storage.sqlite_storage import SQLiteStorage
from src.analysis import transformations as tx

# Cargar las tres series
storage = SQLiteStorage("../data/bcch.db")

df_ipc = storage.load_series("ipc")
df_tpm = storage.load_series("tpm")
df_usd = storage.load_series("usd_clp")

print(f"IPC:     {len(df_ipc):>5} obs   {df_ipc['fecha'].min().date()} → {df_ipc['fecha'].max().date()}")
print(f"TPM:     {len(df_tpm):>5} obs   {df_tpm['fecha'].min().date()} → {df_tpm['fecha'].max().date()}")
print(f"USD/CLP: {len(df_usd):>5} obs   {df_usd['fecha'].min().date()} → {df_usd['fecha'].max().date()}")

IPC:        24 obs   2023-01-01 → 2024-12-01
TPM:       728 obs   2023-01-03 → 2024-12-30
USD/CLP:   728 obs   2023-01-03 → 2024-12-30


In [2]:
df_ipc["mom"]   = tx.pct_change(df_ipc, periods=1)    # variación mensual %
df_ipc["yoy"]   = tx.pct_change(df_ipc, periods=12)   # variación interanual %
df_ipc["ma_3m"] = tx.rolling_mean(df_ipc, window=3, value_col="mom")  # IPC subyacente "casero"

df_ipc.tail(15)


,fecha,valor,mom,yoy,ma_3m
9,2023-10-01,0.4,-42.857143,NaN,160.714286
10,2023-11-01,0.7,75.000000,NaN,210.714286
11,2023-12-01,-0.5,-171.428571,NaN,-46.428571
12,2024-01-01,0.7,-240.000000,-12.500000,-112.142857
13,2024-02-01,0.6,-14.285714,-700.000000,-141.904762
14,2024-03-01,0.4,-33.333333,-63.636364,-95.873016
15,2024-04-01,0.5,25.000000,66.666667,-7.539683
16,2024-05-01,0.3,-40.000000,200.000000,-16.111111
17,2024-06-01,-0.1,-133.333333,-50.000000,-49.444444
18,2024-07-01,0.7,-800.000000,75.000000,-324.444444


In [3]:
df_tpm["diff_pp"] = tx.diff(df_tpm, periods=1)

# Detectar reuniones de política monetaria: días en que la TPM efectivamente cambió
cambios = df_tpm[df_tpm["diff_pp"].abs() > 1e-9].copy()
print(f"Cambios de TPM detectados: {len(cambios)}")
cambios[["fecha", "valor", "diff_pp"]].tail(10)

Cambios de TPM detectados: 9


,fecha,valor,diff_pp
246,2023-09-06,9.50,-0.75
351,2023-12-20,8.25,-0.75
394,2024-02-01,7.25,-1.00
456,2024-04-03,6.50,-0.75
507,2024-05-24,6.00,-0.50
533,2024-06-19,5.75,-0.25
610,2024-09-04,5.50,-0.25
654,2024-10-18,5.25,-0.25
715,2024-12-18,5.00,-0.25


In [5]:
df_usd["ret_1d"]  = tx.pct_change(df_usd, periods=1)
df_usd["vol_30d"] = tx.rolling_std(df_usd, window=30, value_col="ret_1d")
df_usd["ma_30d"]  = tx.rolling_mean(df_usd, window=30)

df_usd.tail(10)

,fecha,valor,ret_1d,vol_30d,ma_30d
718,2024-12-21,NaN,NaN,NaN,NaN
719,2024-12-22,NaN,NaN,NaN,NaN
720,2024-12-23,991.18,NaN,NaN,NaN
721,2024-12-24,992.78,0.161424,NaN,NaN
722,2024-12-25,NaN,NaN,NaN,NaN
723,2024-12-26,989.07,NaN,NaN,NaN
724,2024-12-27,990.95,0.190078,NaN,NaN
725,2024-12-28,NaN,NaN,NaN,NaN
726,2024-12-29,NaN,NaN,NaN,NaN
727,2024-12-30,992.12,NaN,NaN,NaN


In [6]:
# USD/CLP: dos variantes interesantes
df_usd_cierre = tx.to_monthly(df_usd[["fecha", "valor"]], method="last")
df_usd_prom   = tx.to_monthly(df_usd[["fecha", "valor"]], method="mean")

# TPM: el "cierre" es lo natural (qué nivel regía a fin de mes)
df_tpm_mes = tx.to_monthly(df_tpm[["fecha", "valor"]], method="last")

print(f"USD/CLP cierre mensual: {len(df_usd_cierre)} filas")
print(f"TPM mensual:            {len(df_tpm_mes)} filas")
df_usd_cierre.tail(5)

USD/CLP cierre mensual: 24 filas
TPM mensual:            24 filas


,fecha,valor
19,2024-08-01,917.38
20,2024-09-01,896.25
21,2024-10-01,950.89
22,2024-11-01,977.32
23,2024-12-01,992.12


In [7]:
df_wide = tx.merge_wide({
    "ipc":     df_ipc[["fecha", "valor"]],
    "tpm":     df_tpm_mes,
    "usd_clp": df_usd_prom,
})

# Variaciones a nivel del panel mensual
df_wide["ipc_mom"] = df_wide["ipc"].pct_change(1)  * 100
df_wide["ipc_yoy"] = df_wide["ipc"].pct_change(12) * 100
df_wide["usd_yoy"] = df_wide["usd_clp"].pct_change(12) * 100

df_wide.tail(15)

,fecha,ipc,tpm,usd_clp,ipc_mom,ipc_yoy,usd_yoy
9,2023-10-01,0.4,9.00,926.347000,-42.857143,NaN,NaN
10,2023-11-01,0.7,9.00,886.614286,75.000000,NaN,NaN
11,2023-12-01,-0.5,8.25,874.665789,-171.428571,NaN,NaN
12,2024-01-01,0.7,8.25,907.986818,-240.000000,-12.500000,9.881106
13,2024-02-01,0.6,7.25,963.442381,-14.285714,-700.000000,20.693182
14,2024-03-01,0.4,7.25,967.932500,-33.333333,-63.636364,19.571071
15,2024-04-01,0.5,6.50,960.138182,25.000000,66.666667,19.444411
16,2024-05-01,0.3,6.00,917.877143,-40.000000,200.000000,14.930678
17,2024-06-01,-0.1,5.75,926.081053,-133.333333,-50.000000,15.778584
18,2024-07-01,0.7,5.75,937.560909,-800.000000,75.000000,15.264839


In [8]:
cols = ["ipc_yoy", "tpm", "usd_yoy"]
df_wide[cols].corr().round(2)


,ipc_yoy,tpm,usd_yoy
ipc_yoy,1.00,-0.36,-0.42
tpm,-0.36,1.00,0.46
usd_yoy,-0.42,0.46,1.00


In [9]:
# Correlaciones con rezago: ¿el cambio de TPM hoy correlaciona con IPC de los próximos meses?
for lag in range(0, 7):
    corr = df_wide["ipc_yoy"].corr(df_wide["tpm"].shift(lag))
    print(f"lag={lag:>2}m  corr(IPC_yoy, TPM_{lag}m_atras) = {corr:.3f}")


lag= 0m  corr(IPC_yoy, TPM_0m_atras) = -0.356
lag= 1m  corr(IPC_yoy, TPM_1m_atras) = -0.486
lag= 2m  corr(IPC_yoy, TPM_2m_atras) = -0.361
lag= 3m  corr(IPC_yoy, TPM_3m_atras) = -0.438
lag= 4m  corr(IPC_yoy, TPM_4m_atras) = -0.308
lag= 5m  corr(IPC_yoy, TPM_5m_atras) = -0.303
lag= 6m  corr(IPC_yoy, TPM_6m_atras) = -0.350


In [10]:
ultimo = df_ipc.dropna(subset=["yoy"]).iloc[-1]
print(f"Mes más reciente con y/y calculado: {ultimo['fecha'].date()}")
print(f"IPC nivel: {ultimo['valor']:.2f}")
print(f"IPC y/y:   {ultimo['yoy']:.2f}%")


Mes más reciente con y/y calculado: 2024-12-01
IPC nivel: -0.20
IPC y/y:   -60.00%
